# CERES-Wheat Experiment Notebook

End-to-end demonstration of the DSSAT-Python **CERES-Wheat** model (`WHCER048`).

CERES-Wheat extends the CERES cereal framework with two key additions:
- **Vernalization** — cold-temperature induction required by winter wheat (P1V coefficient)
- **Long-day photoperiod response** — flowering delayed by short days (P1D coefficient)

Topics:
1. Experiment JSON definition
2. Winter wheat simulation (vernalization pathway)
3. Spring wheat comparison
4. End-of-season summary
5. Daily trajectory plots
6. Cultivar CUL file I/O
7. Sensitivity analysis — vernalization requirement (P1V)
8. Sowing date × cultivar interaction

## 1 — Setup

In [ ]:
import tempfile, os, pathlib
from dssat.io.converters.cul import write_cul, read_cul

cul_records = [
    {'VAR#': 'IB1500', 'VRNAME': 'Generic Spring Wheat', 'ECO#': 'IWWH01',
     'P1V': 0.0, 'P1D': 3.675, 'P5': 500.0,
     'G1': 6.24, 'G2': 32.0, 'G3': 1.15, 'PHINT': 95.0},
    {'VAR#': 'IB1501', 'VRNAME': 'Generic Winter Wheat', 'ECO#': 'IWWH01',
     'P1V': 50.0, 'P1D': 3.675, 'P5': 550.0,
     'G1': 6.50, 'G2': 35.0, 'G3': 1.20, 'PHINT': 95.0},
]

with tempfile.NamedTemporaryFile(suffix='.CUL', delete=False, mode='w') as f:
    cul_path = f.name

write_cul(cul_records, cul_path, model_code='WH')
print('=== CUL file contents ===')
print(pathlib.Path(cul_path).read_text())

read_back = read_cul(cul_path, model_code='WH')
print('=== Read back ===')
print(json.dumps(read_back, indent=2))
os.unlink(cul_path)

## 2 — Experiment JSON

In [ ]:
experiment = {
    "experiment_id": "WHEXP0001",
    "title": "CERES-Wheat demo — Mediterranean spring wheat",
    "crop": "WH",
    "model": "WHCER048",

    # --- cultivar (IB1500 — generic spring wheat)
    "cultivar": {
        "id": "IB1500",
        "name": "Generic Spring Wheat",
        "ecotype": "IWWH01",
        "p1v":   0.0,
        "p1d":   3.675,
        "p5":  500.0,
        "g1":    6.24,
        "g2":   32.0,
        "g3":    1.15,
        "phint": 95.0,
        "tbase":  0.0,
        "topt":  26.0
    },

    "planting": {
        "date": "2020-11-15",
        "plant_population": 280.0,
        "row_spacing": 20.0,
        "sowing_depth": 3.0
    },

    "simulation": {
        "start_date": "2020-11-15",
        "end_date":   "2021-06-30",
        "water_balance": false,
        "nitrogen_cycle": false
    },

    # --- soil (Mediterranean Vertisol, Tunisia)
    "soil": {
        "id": "TN00000001",
        "name": "Mediterranean Vertisol",
        "country": "TN",
        "layers": [
            {"depth_cm":  15, "bd": 1.25, "ll": 0.18, "dul": 0.34, "sat": 0.48, "oc": 1.20, "ph": 7.5},
            {"depth_cm":  30, "bd": 1.30, "ll": 0.19, "dul": 0.35, "sat": 0.47, "oc": 0.85, "ph": 7.6},
            {"depth_cm":  45, "bd": 1.35, "ll": 0.20, "dul": 0.36, "sat": 0.46, "oc": 0.55, "ph": 7.7},
            {"depth_cm":  60, "bd": 1.40, "ll": 0.21, "dul": 0.37, "sat": 0.45, "oc": 0.35, "ph": 7.8},
            {"depth_cm":  90, "bd": 1.45, "ll": 0.22, "dul": 0.38, "sat": 0.44, "oc": 0.20, "ph": 7.9},
            {"depth_cm": 120, "bd": 1.50, "ll": 0.23, "dul": 0.39, "sat": 0.43, "oc": 0.12, "ph": 8.0}
        ],
        "initial_sw_fraction": 0.7
    }
}

print(json.dumps(experiment, indent=2)[:600], '...')

## 3 — Build and run the simulation

CERES-Wheat uses a **Mediterranean seasonal pattern**: mild wet winters → warm dry spring.

In [ ]:
# Build soil
soil = SoilType()
layers = experiment['soil']['layers']
soil.nlayr = len(layers)
depth_cum = 0.0
for i, lyr in enumerate(layers):
    thick = lyr['depth_cm'] - depth_cum
    soil.dlayr[i] = thick
    soil.ds[i]    = lyr['depth_cm']
    soil.ll[i]    = lyr['ll']
    soil.dul[i]   = lyr['dul']
    soil.sat[i]   = lyr['sat']
    soil.bd[i]    = lyr['bd']
    soil.shf[i]   = 1.0
    soil.kg2ppm[i]= 10.0 / (lyr['bd'] * thick)
    depth_cum     = lyr['depth_cm']

sw_full = np.zeros(NL)
sw_init = 0.7
sw_full[:soil.nlayr] = [soil.dul[i]*sw_init + soil.ll[i]*(1-sw_init) for i in range(soil.nlayr)]

# Cultivar
cv_dict = experiment['cultivar']
cultivar = WheatCultivar(
    id=cv_dict.get('id', 'IB1500'),
    name=cv_dict.get('name', 'Generic Spring Wheat'),
    p1v=cv_dict['p1v'], p1d=cv_dict['p1d'], p5=cv_dict['p5'],
    g1=cv_dict['g1'], g2=cv_dict['g2'], g3=cv_dict['g3'],
    phint=cv_dict['phint'], tbase=cv_dict['tbase'], topt=cv_dict['topt'],
)

yrsim = 2020320  # 2020-11-15 ≈ DOY 320
pltg = experiment['planting']

model = CeresWheat(
    cultivar=cultivar,
    pltpop=pltg['plant_population'],
    sdepth=pltg['sowing_depth'],
    yrsim=yrsim,
    yrplt=yrsim,
)

iswitch = SwitchType()
iswitch.iswwat = 'N'
iswitch.iswnit = 'N'

ctrl = ControlType()
ctrl.yrsim = yrsim

def med_weather(doy: int) -> WeatherType:
    """Synthetic Mediterranean seasonal weather."""
    # Normalize DOY to 0–365 range relative to sowing (DOY 320)
    d_rel = (doy - 320) % 365
    # Temperature: cold in winter (Jan-Feb), warm in spring (Apr-May)
    tmax = 16.0 + 14.0 * np.sin((d_rel - 40) * np.pi / 180)
    tmin = 6.0  +  8.0 * np.sin((d_rel - 40) * np.pi / 180)
    srad = 12.0 +  8.0 * np.sin((d_rel - 20) * np.pi / 180)
    dayl = 10.0 +  4.0 * np.sin((d_rel - 50) * np.pi / 180)
    w = WeatherType()
    w.tmax = max(5, tmax); w.tmin = max(-2, tmin)
    w.srad = max(5, srad); w.dayl = max(8, min(16, dayl))
    w.co2 = 410.0; w.tavg = (w.tmax + w.tmin) / 2
    w.rain = 0.0; w.snow = max(0.0, -w.tmin * 0.5)
    return w

w0 = med_weather(320)
ctrl.dynamic = RUNINIT; ctrl.yrdoy = yrsim
model.run(ctrl, iswitch, soil, w0, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
ctrl.dynamic = SEASINIT
model.run(ctrl, iswitch, soil, w0, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)

records = []
for day in range(1, 280):
    yrdoy = incdat(yrsim, day)
    ctrl.yrdoy = yrdoy
    doy = yrdoy % 1000
    w = med_weather(doy)

    ctrl.dynamic = RATE
    model.run(ctrl, iswitch, soil, w, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
    ctrl.dynamic = INTEGR
    model.run(ctrl, iswitch, soil, w, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)

    g = model.growth
    p = model.pheno
    records.append({
        'day': day, 'yrdoy': yrdoy, 'doy': doy,
        'istage': p.istage, 'xstage': p.xstage,
        'lai': g.lai, 'biomas': g.biomas, 'yield_kg_ha': g.yield_,
        'tmax': w.tmax, 'tmin': w.tmin, 'srad': w.srad, 'dayl': w.dayl,
    })
    if p.istage == 6 and p.mdate > 0:
        print(f'Maturity: day {day} (YRDOY {yrdoy})')
        break

df = pd.DataFrame(records)
print(f'{len(df)} days simulated')
print(df[['day','istage','lai','biomas','yield_kg_ha']].tail(8).to_string(index=False))

## 4 — End-of-season summary

In [ ]:
summary = model.summary()
print('\n=== CERES-Wheat End-of-Season Summary ===')
for k, v in summary.items():
    print(f'  {k:<25} {v}')

## 5 — Daily trajectory plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('CERES-Wheat — Seasonal trajectories', fontsize=13, fontweight='bold')

stage_labels = {1:'Emg',2:'Hdg',3:'Anthi',4:'Gfill',5:'End GF',6:'Mat'}
stage_colors = {'1':'#A8D5A2','2':'#F9E07C','3':'#F4A460','4':'#DEB887','5':'#DAA520','6':'#CD853F'}

# Stage transitions as vertical bands
ax = axes[0, 0]
ax.plot(df['day'], df['xstage'], color='navy', lw=2)
ax.set_ylabel('Growth stage'); ax.set_xlabel('Days after sowing')
ax.set_title('Phenological development'); ax.set_ylim(0, 7)
for stage, label in stage_labels.items():
    rows = df[df['istage'] == int(stage)]
    if not rows.empty:
        ax.axvline(rows['day'].iloc[0], color='grey', ls=':', alpha=0.7)
        ax.text(rows['day'].iloc[0]+0.5, 0.3, label, fontsize=8, rotation=90, va='bottom', color='grey')
ax.grid(True, alpha=0.3)

# Temperature + vernalization context
ax = axes[0, 1]
ax.fill_between(df['day'], df['tmin'], df['tmax'], alpha=0.3, color='steelblue', label='Temp range')
ax.plot(df['day'], (df['tmax']+df['tmin'])/2, 'steelblue', lw=1.5, label='Tmean')
ax.axhline(0, color='navy', ls='--', lw=1, alpha=0.5, label='0°C')
ax.axhline(5, color='purple', ls='--', lw=1, alpha=0.5, label='5°C (vern threshold)')
ax.set_ylabel('Temperature (°C)'); ax.set_xlabel('Days after sowing')
ax.set_title('Temperature and vernalization window')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# LAI
ax = axes[1, 0]
ax.plot(df['day'], df['lai'], color='forestgreen', lw=2)
ax.set_ylabel('LAI (m² m⁻²)'); ax.set_xlabel('Days after sowing')
ax.set_title('Leaf area index'); ax.grid(True, alpha=0.3)

# Biomass and yield
ax = axes[1, 1]
ax.plot(df['day'], df['biomas'], color='saddlebrown', lw=2, label='Biomass (g m⁻²)')
ax2 = ax.twinx()
ax2.plot(df['day'], df['yield_kg_ha'], color='goldenrod', lw=2, ls='--', label='Grain yield')
ax.set_ylabel('Biomass (g m⁻²)', color='saddlebrown')
ax2.set_ylabel('Grain yield (kg ha⁻¹)', color='goldenrod')
ax.set_xlabel('Days after sowing'); ax.set_title('Biomass and grain yield')
lines1, l1 = ax.get_legend_handles_labels()
lines2, l2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, l1+l2, loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('wheat_trajectories.png', bbox_inches='tight')
plt.show()

## 6 — Cultivar CUL file I/O

In [ ]:
import tempfile, os, pathlib
from dssat.io.converters.cul import write_cul, read_cul

cul_records = [
    {'VAR#': 'IB1500', 'VRNAME': 'Generic Spring Wheat', 'ECO#': 'IWWH01',
     'P1V': 0.0, 'P1D': 3.675, 'P5': 500.0,
     'G1': 6.24, 'G2': 32.0, 'G3': 1.15, 'PHINT': 95.0},
    {'VAR#': 'IB1501', 'VRNAME': 'Generic Winter Wheat', 'ECO#': 'IWWH01',
     'P1V': 50.0, 'P1D': 3.675, 'P5': 550.0,
     'G1': 6.50, 'G2': 35.0, 'G3': 1.20, 'PHINT': 95.0},
]

with tempfile.NamedTemporaryFile(suffix='.CUL', delete=False, mode='w') as f:
    cul_path = f.name

write_cul(cul_records, cul_path, model_code='WH')
print('=== CUL file contents ===')
print(pathlib.Path(cul_path).read_text())

read_back = read_cul(cul_path, model_code='WH')
print('=== Read back ===')
print(json.dumps(read_back, indent=2))
os.unlink(cul_path)

## 7 — Sensitivity to vernalization requirement (P1V)

P1V = 0 → obligate spring type (no vernalization needed)  
P1V > 0 → facultative or winter type; larger values require more vernalization days

In [ ]:
p1v_values = [0.0, 10.0, 20.0, 30.0, 50.0, 70.0]
sens_results = []

for p1v in p1v_values:
    cv = WheatCultivar(p1v=p1v, p1d=3.675, p5=500.0, g1=6.24, g2=32.0, g3=1.15, phint=95.0)
    m = CeresWheat(cultivar=cv, pltpop=280.0, sdepth=3.0, yrsim=yrsim, yrplt=yrsim)

    ctrl5 = ControlType(); ctrl5.yrsim = yrsim
    ctrl5.dynamic = RUNINIT; ctrl5.yrdoy = yrsim
    m.run(ctrl5, iswitch, soil, w0, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
    ctrl5.dynamic = SEASINIT
    m.run(ctrl5, iswitch, soil, w0, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)

    das_heading = None
    for day in range(1, 280):
        yrdoy = incdat(yrsim, day)
        ctrl5.yrdoy = yrdoy
        doy = yrdoy % 1000
        w_ = med_weather(doy)
        ctrl5.dynamic = RATE
        m.run(ctrl5, iswitch, soil, w_, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
        ctrl5.dynamic = INTEGR
        m.run(ctrl5, iswitch, soil, w_, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
        if m.pheno.istage >= 3 and das_heading is None:
            das_heading = day
        if m.pheno.istage == 6 and m.pheno.mdate > 0:
            break

    s = m.summary()
    res = {
        'p1v': p1v,
        'yield_kg_ha': s['yield_kg_ha'],
        'das_heading': das_heading or day,
        'das_maturity': day,
    }
    sens_results.append(res)
    print(f'P1V={p1v:5.1f}  heading=day {res["das_heading"]:3d}  maturity=day {day:3d}  yield={s["yield_kg_ha"]:7.1f} kg/ha')

df_p1v = pd.DataFrame(sens_results)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Wheat sensitivity to vernalization requirement (P1V)', fontsize=12, fontweight='bold')

axes[0].plot(df_p1v['p1v'], df_p1v['das_heading'], 'o-', color='steelblue', lw=2, ms=8, label='Heading')
axes[0].plot(df_p1v['p1v'], df_p1v['das_maturity'], 's--', color='saddlebrown', lw=2, ms=8, label='Maturity')
axes[0].set_xlabel('P1V (vernalization days)'); axes[0].set_ylabel('Days after sowing')
axes[0].set_title('Days to heading and maturity'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(df_p1v['p1v'], df_p1v['yield_kg_ha'], 'o-', color='goldenrod', lw=2, ms=8)
axes[1].set_xlabel('P1V (vernalization days)'); axes[1].set_ylabel('Grain yield (kg ha⁻¹)')
axes[1].set_title('Yield vs vernalization requirement'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('wheat_p1v_sensitivity.png', bbox_inches='tight')
plt.show()

## 8 — Sowing date × cultivar interaction

Compare spring wheat (P1V=0) vs facultative wheat (P1V=30) across sowing dates.

In [ ]:
sow_doys = [290, 305, 320, 335, 350, 365]  # Oct-15 to Dec-31
sow_labels = ['Oct-17', 'Nov-1', 'Nov-16', 'Dec-1', 'Dec-16', 'Dec-31']

cultivar_specs = [
    ('Spring (P1V=0)',       WheatCultivar(p1v=0.0,  p1d=3.675, p5=500, g1=6.24, g2=32.0, g3=1.15, phint=95.0)),
    ('Facultative (P1V=30)', WheatCultivar(p1v=30.0, p1d=3.675, p5=500, g1=6.24, g2=32.0, g3=1.15, phint=95.0)),
]

interaction = []
for cv_name, cv_spec in cultivar_specs:
    for sow_doy, sow_lbl in zip(sow_doys, sow_labels):
        yrsim_s = 2020000 + sow_doy
        m = CeresWheat(cultivar=cv_spec, pltpop=280.0, sdepth=3.0, yrsim=yrsim_s, yrplt=yrsim_s)
        ctrl6 = ControlType(); ctrl6.yrsim = yrsim_s
        ctrl6.dynamic = RUNINIT; ctrl6.yrdoy = yrsim_s
        w_s = med_weather(sow_doy)
        m.run(ctrl6, iswitch, soil, w_s, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
        ctrl6.dynamic = SEASINIT
        m.run(ctrl6, iswitch, soil, w_s, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)

        for day in range(1, 300):
            yrdoy = incdat(yrsim_s, day)
            ctrl6.yrdoy = yrdoy
            doy = yrdoy % 1000
            w_ = med_weather(doy)
            ctrl6.dynamic = RATE
            m.run(ctrl6, iswitch, soil, w_, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
            ctrl6.dynamic = INTEGR
            m.run(ctrl6, iswitch, soil, w_, sw_full, np.zeros(NL), np.zeros(NL), 0, 0)
            if m.pheno.istage == 6 and m.pheno.mdate > 0:
                break

        s = m.summary()
        interaction.append({
            'cultivar': cv_name, 'sow_doy': sow_doy,
            'sow_label': sow_lbl, 'yield_kg_ha': s['yield_kg_ha'],
        })

df_int = pd.DataFrame(interaction)
print(df_int.pivot(index='sow_label', columns='cultivar', values='yield_kg_ha').to_string())

fig, ax = plt.subplots(figsize=(9, 4))
colors_cv = ['steelblue', 'darkorange']
for (cv_name, _), color in zip(cultivar_specs, colors_cv):
    sub = df_int[df_int['cultivar'] == cv_name]
    ax.plot(sub['sow_doy'], sub['yield_kg_ha'], 'o-', color=color, lw=2, ms=8, label=cv_name)
ax.set_xlabel('Sowing DOY (2020)')
ax.set_ylabel('Grain yield (kg ha⁻¹)')
ax.set_title('Wheat yield × sowing date × vernalization type')
ax.set_xticks(sow_doys); ax.set_xticklabels(sow_labels, rotation=30)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('wheat_sowdate_cultivar.png', bbox_inches='tight')
plt.show()